# ELEVATION

In [ ]:
import ee
import os
import io
import requests
import numpy as np
import rasterio

def process_gee_task(task, imagecollection, data_name):
    # 1. Initialize Earth Engine
    try:
        ee.Initialize()
    except Exception as e:
        print("Earth Engine not initialized. Attempting to authenticate...")
        ee.Authenticate()
        ee.Initialize()

    event_id = task['event_id']
    
    print(f"Processing Elevation for Event: {task['name']} ({event_id})")

    # 2. Define Geometry
    bounds = task['target_bounds']
    roi = ee.Geometry.Rectangle([bounds[0], bounds[1], bounds[2], bounds[3]], proj=task['crs'], geodesic=False)
    
    # 3. Select Data
    # 1m data is an ImageCollection, so we mosaic it to get a single seamless image.
    elev_collection = ee.ImageCollection(imagecollection)
    elev_image = elev_collection.filterBounds(roi).mosaic().resample('bilinear') # TODO: check
    
    # 4. Prepare Download URL
    # We specify the CRS, Scale, and Region here. GEE handles the 
    # reprojection (Resampling) and Rasterization server-side before download.
    download_params = {
        'name': f'{data_name}_{event_id}',
        'crs': task['crs'],
        'scale': task['resolution'],
        'region': roi, # The download region is the original geometry
        'format': 'GEO_TIFF'
    }

    try:
        url = elev_image.getDownloadURL(download_params)
        print(f"Download URL generated. Fetching data...")
        print(url)
    except Exception as e:
        print(f"Error generating download URL (Area might be too large): {e}")
        return

    # 5. Download and Read into Memory
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"HTTP Error {response.status_code}: {response.text}")
        response.raise_for_status()

    # Check content type to handle both ZIP (default) and raw TIFF (small areas)
    content_type = response.headers.get('Content-Type', '')
    print(f"Received content type: {content_type}")

    try:
        # It is a raw GeoTIFF
        with rasterio.open(io.BytesIO(response.content)) as src:
            data = src.read(1)
                    
    except (rasterio.errors.RasterioIOError):
        print("\n--- ERROR DOWNLOADING DATA ---")
        print("The server returned a file that could not be processed as TIFF.")
        print(f"Content-Type received: {response.headers.get('Content-Type')}")
        print("Response content (First 500 bytes):")
        print(response.content[:500])
        print("------------------------------")
        raise

    # 6. Format Output
    # The prompt requests a specific dictionary structure.
    output_payload = {
        'data': data, # The numpy array
        'timestamps': [fire_info['t_start']], # Using start time as the anchor timestamp
        'metadata': {
            'name': data_name,
            'crs': task['crs'],
            'resolution': task['resolution'],
            'source': imagecollection,
        }
    }

    # 7. Save to File
    output_dir = os.path.join('./output', event_id)
    os.makedirs(output_dir, exist_ok=True)
    
    save_path = os.path.join(output_dir, f'{data_name}.npy')
    
    # Save as a pickled numpy dictionary
    np.save(save_path, output_payload)
    
    print(f"Success. Data shape: {data.shape}")
    print(f"Saved to: {save_path}")


In [ ]:
def process_elevation_task(task):
    return process_gee_task(task, 'USGS/3DEP/1m', data_name='elevation')


In [ ]:
process_elevation_task(task)

# LANDFIRE

In [ ]:
def process_cbd_task(task):
    process_gee_task(task, 'projects/sat-io/open-datasets/landfire/FUEL/CBD', data_name='cbd')


In [ ]:
process_cbd_task(task)

In [ ]:
def process_cc_task(task):
    process_gee_task(task, 'projects/sat-io/open-datasets/landfire/FUEL/CC', data_name='cc')


In [ ]:
process_cc_task(task)

# HRRR

In [ ]:
import os
import datetime
import numpy as np
import xarray as xr
import rioxarray
import pyproj
from herbie import Herbie # TODO: remove after downloaded to home.
import pandas as pd

# ---------------------------------------------------------
# 1. Helper Functions (Coordinate Fix & Geospatial Processing)
# ---------------------------------------------------------

def fix_hrrr_coords(ds):
    """
    Calculates 1D x/y coordinates in meters from the 2D lat/lon arrays 
    and assigns them to the dataset so rioxarray can read the grid.
    """
    # 1. Ensure we have the CRS object loaded from metadata
    if ds.rio.crs is None:
        try:
            # HRRR usually has a variable like 'gribfile_projection' or 'unknown'
            proj_var = [v for v in ds.data_vars if 'projection' in v]
            if proj_var:
                ds = ds.rio.write_crs(ds[proj_var[0]].attrs)
        except Exception:
            pass

    if ds.rio.crs is None:
        # Fallback for HRRR (Lambert Conformal) if metadata fails
        # This is the standard HRRR projection parameter set
        ds = ds.rio.write_crs("+proj=lcc +lat_1=38.5 +lat_2=38.5 +lat_0=38.5 +lon_0=-97.5 +x_0=0 +y_0=0 +a=6371229 +b=6371229 +units=m +no_defs")

    # 2. Create a transformer from Lat/Lon (EPSG:4326) to the HRRR Projected CRS
    transformer = pyproj.Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)

    # 3. Transform the 2D Lat/Lon arrays to Projected X/Y
    # HRRR grib imports usually result in 'latitude' and 'longitude' coordinates
    xx, yy = transformer.transform(ds.longitude.values, ds.latitude.values)

    # 4. Extract 1D vectors
    # Since HRRR is a rectilinear grid in its own projection, 
    # x is constant along columns, y is constant along rows.
    x_1d = xx[0, :]
    y_1d = yy[:, 0]

    # 5. Assign these new coordinates to the x and y dimensions
    # Rename dimensions if they are strictly 'y' and 'x' in the GRIB import
    ds = ds.assign_coords(x=x_1d, y=y_1d)
    
    return ds

def process_geospatial(ds, target_crs, bounds):
    """
    Standardizes coords, reprojects, and clips the dataset.
    """
    # Standardize Dimension Names if needed (grib often loads as y, x)
    if 'latitude' in ds.dims and 'longitude' in ds.dims:
        ds = ds.rename({'longitude': 'x', 'latitude': 'y'})
    
    # Fix Coordinates
    ds = fix_hrrr_coords(ds)
    
    # Reproject
    ds_reproj = ds.rio.reproject(target_crs)

    # Crop
    ds_cropped = ds_reproj.rio.clip_box(
        minx=bounds[0],
        miny=bounds[1],
        maxx=bounds[2],
        maxy=bounds[3]
    )
    
    return ds_cropped

# ---------------------------------------------------------
# 2. Main Processing Logic
# ---------------------------------------------------------

def process_hrrr_task(task):
    print(f"Processing Task: {task['name']} ({task['event_id']})")
    
    # Setup Output Directory
    output_dir = f"./output/{task['event_id']}/"
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize storage lists for aggregation
    data_buffer = {
        'r2': [],  # Humidity
        'u10': [], # Wind U
        'v10': []  # Wind V
    }
    timestamps = []

    # Generate list of hours to process
    # Creates a range from start to end (inclusive)
    current_time = task['t_start']
    end_time = task['t_end']
    
    while current_time <= end_time:
        print(f"Downloading HRRR for: {current_time}")
        
        try:
            # Initialize Herbie
            H = Herbie(
                current_time,
                model='hrrr',
                product='sfc',
                fxx=0,
                verbose=False # Set to True if you want Herbie logs
            )
            
            # --- 1. Process Humidity ---
            ds_rh = H.xarray(":RH:2 m", remove_grib=False)
            ds_rh_proc = process_geospatial(ds_rh, task['crs'], task['target_bounds'])
            
            # --- 2. Process Wind ---
            ds_wind = H.xarray(":(?:UGRD|VGRD):10 m", remove_grib=False)
            ds_wind_proc = process_geospatial(ds_wind, task['crs'], task['target_bounds'])
            
            # Append data to buffers
            # We use .values to get the numpy array of the 2D slice
            # variable names usually match the cfgrib shortNames
            data_buffer['r2'].append(ds_rh_proc['r2'].values)
            data_buffer['u10'].append(ds_wind_proc['u10'].values)
            data_buffer['v10'].append(ds_wind_proc['v10'].values)
            
            timestamps.append(current_time)

        except Exception as e:
            print(f"Failed to process {current_time}: {e}")
        
        # Increment time by 1 hour
        current_time += datetime.timedelta(hours=1)

    # ---------------------------------------------------------
    # 3. Aggregation and Saving
    # ---------------------------------------------------------
    
    if not timestamps:
        print("No data processed.")
        return

    # Convert timestamp list to numpy array of datetime64
    ts_array = np.array(timestamps, dtype='datetime64[ns]')

    # Save function to keep things clean
    def save_variable(var_name, save_name):
        # Stack list of 2D arrays into a 3D array (Time, Y, X)
        stacked_data = np.stack(data_buffer[var_name], axis=0)
        
        output_dict = {
            'data': stacked_data,
            'timestamps': ts_array,
            'metadata': {
                'name': 'FEDS25MTBS'
            }
        }
        
        filename = os.path.join(output_dir, save_name)
        np.save(filename, output_dict)
        print(f"Saved {save_name} with shape {stacked_data.shape}")

    # Execute Saves
    save_variable('r2', 'humidity.npy')
    save_variable('u10', 'wind_u.npy')
    save_variable('v10', 'wind_v.npy')

    print(f"Task completed. Output saved to {output_dir}")


In [ ]:
# Run
process_hrrr_task(task)

# Urban Data

In [ ]:
# Height, TBD from GEE
# LAI waiting for publish, but already got.
# building height


# https://github.com/zhu-xlab/GlobalBuildingAtlas

In [ ]:
import geopandas as gpd
import numpy as np
import plotly.express as px


def visualize_fire(fire_data: dict):
    
    data_dict = {
        'timestamp': fire_data.get('timestamps'),
        'geometry': fire_data.get('data')
    }

    gdf = gpd.GeoDataFrame(data_dict, geometry='geometry')

    gdf = gdf.sort_values('timestamp')
    gdf['Time'] = gdf['timestamp'].dt.strftime('%Y-%m-%d %H:%M')
    gdf['Status'] = 'Active Fire'

    minx, miny, maxx, maxy = gdf.total_bounds
    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2
    
    max_bound_diff = max(maxx - minx, maxy - miny)
    zoom_level = 9.5 - np.log(max_bound_diff) if max_bound_diff > 0 else 10

    fig = px.choropleth_map(
        gdf,
        geojson=gdf.geometry,
        locations=gdf.index,
        color='Status',
        animation_frame='Time',
        color_discrete_map={'Active Fire': '#FF4500'},
        opacity=0.6,
        center={"lat": center_lat, "lon": center_lon},
        zoom=zoom_level,
        title=f"Fire Progression: {fire_data.get('metadata').get('name')}"
    )
    
    fig.update_layout(
        map_style="white-bg",
        map=dict(
            center={"lat": center_lat, "lon": center_lon},
            zoom=zoom_level,
            layers=[
                {
                    "below": 'traces',
                    "sourcetype": "raster",
                    "sourceattribution": "Esri, Maxar, Earthstar Geographics",
                    "source": [
                        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
                    ]
                }
            ]
        ),
        height=800, 
        margin={"r":0,"t":40,"l":0,"b":0},
        transition={'duration': 0}
    )

    fig.show()


In [ ]:
visualize_fire(extract_fire_data(fire_info))